# Laboratório — Variáveis aleatórias, PMF, PDF e CDF

Este laboratório reproduz os principais objetos da Aula 05 e verifica resultados por testes automáticos.

**Dependências:** Python 3.10+, NumPy 1.24+, Matplotlib 3.7+ e SciPy 1.10+.  
**Reprodutibilidade:** todas as amostras usam `numpy.random.default_rng(20260907)`.


## 1. Preparação

Execute as células em ordem. As versões são exibidas para tornar o ambiente auditável.


In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import stats

SEED = 20260907
rng = np.random.default_rng(SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Seed: {SEED}")


## 2. PMF do número de caras

Em dois lançamentos de uma moeda justa, a variável $X$ conta caras e tem suporte $\{0,1,2\}$.


In [ ]:
x = np.array([0, 1, 2])
pmf = np.array([0.25, 0.50, 0.25])

assert np.all(pmf >= 0)
assert np.isclose(pmf.sum(), 1.0)

p_ao_menos_uma = pmf[x >= 1].sum()
assert np.isclose(p_ao_menos_uma, 0.75)
print(f"P(X ≥ 1) exata = {p_ao_menos_uma:.4f}")


Uma simulação não prova a PMF, mas funciona como teste de coerência. Geramos os dois lançamentos diretamente, sem recorrer a uma distribuição pronta.


In [ ]:
n = 200_000
lancamentos = rng.integers(0, 2, size=(n, 2))  # 1 = cara
amostra_x = lancamentos.sum(axis=1)
frequencias = np.bincount(amostra_x, minlength=3) / n

assert np.allclose(frequencias, pmf, atol=0.004)
print("PMF exata:     ", pmf)
print("Frequência sim.:", np.round(frequencias, 6))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
largura = 0.34
ax.bar(x - largura/2, pmf, largura, label="PMF exata", color="#2563eb")
ax.bar(x + largura/2, frequencias, largura, label="Frequência simulada", color="#f59e0b")
ax.set(xlabel="Número de caras X", ylabel="Probabilidade / frequência",
       title="Distribuição do número de caras em dois lançamentos", xticks=x, ylim=(0, 0.6))
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 3. CDF discreta

A soma cumulativa da PMF gera $F_X(x)=P(X\leq x)$. A massa de cada ponto é o tamanho do salto.


In [ ]:
cdf = np.cumsum(pmf)
assert np.all(np.diff(cdf) >= 0)
assert np.isclose(cdf[-1], 1.0)
assert np.allclose(np.diff(np.r_[0, cdf]), pmf)

for valor, acumulada in zip(x, cdf):
    print(f"F({valor}) = {acumulada:.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.step(np.r_[-0.5, x, 2.5], np.r_[0, cdf, 1], where="post", color="#7c3aed")
ax.scatter(x, cdf, color="#7c3aed", zorder=3)
ax.set(xlabel="x", ylabel="F(x)", title="CDF discreta: os saltos são as massas",
       xlim=(-0.5, 2.5), ylim=(-0.03, 1.05))
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 4. PDF e CDF de uma variável contínua

Modele a latência $T$ como uniforme em $[0,4]$. A PDF vale $1/4$ no suporte, e a CDF cresce linearmente de 0 a 1.


In [ ]:
latencia = stats.uniform(loc=0, scale=4)
a, b = 1.0, 2.5
p_intervalo = latencia.cdf(b) - latencia.cdf(a)

assert np.isclose(latencia.pdf(2.0), 0.25)
assert np.isclose(p_intervalo, 0.375)
assert np.isclose(latencia.cdf(4.0), 1.0)
print(f"f_T(2) = {latencia.pdf(2.0):.3f} por segundo")
print(f"P(1 < T ≤ 2,5) = {p_intervalo:.3f}")
print("P(T = 2) = 0 no modelo contínuo")


In [ ]:
grade = np.linspace(-0.5, 4.5, 501)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(grade, latencia.pdf(grade), color="#0f766e", lw=2)
mascara = (grade >= a) & (grade <= b)
ax1.fill_between(grade[mascara], latencia.pdf(grade[mascara]), alpha=0.35, color="#14b8a6",
                 label="área = 0,375")
ax1.set(xlabel="Latência (s)", ylabel="Densidade (1/s)", title="PDF: probabilidade é área")
ax1.legend()
ax1.grid(alpha=0.25)

ax2.plot(grade, latencia.cdf(grade), color="#be123c", lw=2)
ax2.set(xlabel="Latência (s)", ylabel="F(t)", title="CDF: probabilidade acumulada", ylim=(-0.03, 1.03))
ax2.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 5. Uma PDF pode ser maior que 1

Na uniforme em $[0,0{,}5]$, a densidade é 2, mas a área total continua igual a 1.


In [ ]:
u_curta = stats.uniform(loc=0, scale=0.5)
altura = u_curta.pdf(0.25)
p_faixa = u_curta.cdf(0.2) - u_curta.cdf(0.1)

assert np.isclose(altura, 2.0)
assert np.isclose(p_faixa, 0.2)
print(f"Densidade em 0,25: {altura:.1f}")
print(f"P(0,1 < U ≤ 0,2): {p_faixa:.1f}")
print(f"Área total: {altura * 0.5:.1f}")


## 6. CDF empírica

A CDF empírica é a fração de observações menores ou iguais a cada limiar. Mesmo para dados contínuos, ela é uma função em degraus.


In [ ]:
amostra_t = latencia.rvs(size=5_000, random_state=rng)
ordenada = np.sort(amostra_t)
ecdf = np.arange(1, len(ordenada) + 1) / len(ordenada)

assert np.all((amostra_t >= 0) & (amostra_t <= 4))
assert np.all(np.diff(ecdf) > 0)
assert np.isclose(ecdf[-1], 1.0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.step(ordenada, ecdf, where="post", label="CDF empírica (n=5.000)", color="#d97706")
ax.plot(grade, latencia.cdf(grade), "--", label="CDF teórica", color="#1d4ed8")
ax.set(xlabel="Latência (s)", ylabel="Probabilidade acumulada", title="CDF empírica × CDF teórica",
       xlim=(-0.1, 4.1), ylim=(-0.03, 1.03))
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 7. Amostragem por transformação inversa

Para a uniforme em $[0,4]$, $F(t)=t/4$ e, portanto, $F^{-1}(u)=4u$. Verificamos a amostra por duas consultas à CDF.


In [ ]:
u = rng.random(100_000)
t_inversa = 4 * u
f1_emp = np.mean(t_inversa <= 1)
f3_emp = np.mean(t_inversa <= 3)

assert np.all((t_inversa >= 0) & (t_inversa < 4))
assert np.isclose(f1_emp, 0.25, atol=0.005)
assert np.isclose(f3_emp, 0.75, atol=0.005)
print(f"F̂(1) = {f1_emp:.6f}; F(1) = 0.250000")
print(f"F̂(3) = {f3_emp:.6f}; F(3) = 0.750000")


## 8. Desafio curto

Altere o intervalo uniforme para $[2,5]$ sem modificar a seed. Antes de executar, preveja:

1. a altura da PDF;
2. $P(3\leq Y\leq4{,}5)$;
3. os valores da CDF em 2, 3 e 5.

**Resposta esperada:** densidade $1/3$; probabilidade $0{,}5$; CDFs 0, $1/3$ e 1. Use `stats.uniform(loc=2, scale=3)` como fallback executável.


In [ ]:
desafio = stats.uniform(loc=2, scale=3)
assert np.isclose(desafio.pdf(3), 1/3)
assert np.isclose(desafio.cdf(4.5) - desafio.cdf(3), 0.5)
assert np.allclose(desafio.cdf([2, 3, 5]), [0, 1/3, 1])
print("Desafio validado: PDF = 1/3; probabilidade = 0,5; CDF = [0, 1/3, 1].")


## Conclusão

O laboratório confirmou que massas são somadas, densidades são integradas e CDFs unificam os dois casos. Também distinguiu distribuição teórica de estimativas empíricas. Na Aula 06, essas distribuições serão usadas para calcular esperança, variância e covariância.
